# Direct Preference Optimization (DPO) Experiment

**Model:** `HuggingFaceTB/SmolLM2-360M`  
**Dataset:** Anthropic HH-RLHF

This notebook implements the Direct Preference Optimization component of the term-paper experiment comparing **Supervised Fine-Tuning (SFT)** and **Direct Preference Optimization (DPO)** for preference alignment in a compact language model.

The workflow covers environment setup, preference-data preparation, DPO training, model saving, and held-out preference evaluation.


## 1. Install Dependencies

Install the library versions used for the DPO experiment while retaining the Colab PyTorch installation.


In [1]:
# Install only the libraries required for this DPO experiment.
# Do not install/upgrade torch, torchvision, torchaudio, CUDA, or NumPy:
# Colab supplies a mutually compatible PyTorch/CUDA stack.
%pip install -q \
    "transformers==4.53.3" \
    "datasets==3.6.0" \
    "accelerate==1.8.1" \
    "trl==0.19.1" \
    "huggingface-hub==0.36.2" \
    "fsspec==2025.3.0"

print("✓ DPO experiment dependencies installed")
print("PyTorch/CUDA packages were left unchanged.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires 

## 2. Verify the Environment

Verify the main package versions, CUDA availability, GPU, and TRL DPO classes before running the experiment.


In [2]:
import torch
import transformers
import trl
import datasets
import accelerate

print("=" * 60)
print("DPO ENVIRONMENT")
print("=" * 60)

print("Torch         :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("TRL           :", trl.__version__)
print("Datasets      :", datasets.__version__)
print("Accelerate    :", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from trl import DPOTrainer, DPOConfig

print("\n✓ DPOTrainer imported successfully")
print("✓ DPOConfig imported successfully")


DPO ENVIRONMENT
Torch         : 2.11.0+cu128
Transformers  : 4.53.3
TRL           : 0.19.1
Datasets      : 3.6.0
Accelerate    : 1.8.1

CUDA available: True
GPU: Tesla T4

✓ DPOTrainer imported successfully
✓ DPOConfig imported successfully


## 3. Imports and Reproducibility

Import the required libraries and fix the random seed used for sampling and training.


In [3]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from trl import DPOTrainer, DPOConfig

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("ENVIRONMENT READY")
print("=" * 60)

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSeed:", SEED)


ENVIRONMENT READY
Device: cuda
GPU: Tesla T4

Seed: 42


## 4. Load the HH-RLHF Dataset

Load Anthropic's HH-RLHF dataset containing human preference pairs.


In [4]:
DATASET_NAME = "Anthropic/hh-rlhf"

dataset = load_dataset(DATASET_NAME)

print("=" * 60)
print("HH-RLHF DATASET")
print("=" * 60)

print(dataset)

print("\nTraining examples :", len(dataset["train"]))
print("Test examples     :", len(dataset["test"]))

print("\nColumns:")
print(dataset["train"].column_names)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

HH-RLHF DATASET
DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 160800
    })
    test: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 8552
    })
})

Training examples : 160800
Test examples     : 8552

Columns:
['chosen', 'rejected']


## 5. Create Experimental Subsets

Sample 1,000 training preference pairs and 199 held-out test pairs using the fixed random seed.


In [5]:
TRAIN_SIZE = 1000
TEST_SIZE = 199

train_data = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE))
)

test_data = (
    dataset["test"]
    .shuffle(seed=SEED)
    .select(range(TEST_SIZE))
)

print("=" * 60)
print("SAMPLED DATASET")
print("=" * 60)

print(f"Training pairs : {len(train_data)}")
print(f"Test pairs     : {len(test_data)}")

print("\nExample keys:")
print(train_data.column_names)

print("\nChosen preview:\n")
print(train_data[0]["chosen"][:300])

print("\nRejected preview:\n")
print(train_data[0]["rejected"][:300])


SAMPLED DATASET
Training pairs : 1000
Test pairs     : 199

Example keys:
['chosen', 'rejected']

Chosen preview:



Human: Why did cells originally combine together to create life?

Assistant: Because their simple components -- chemicals -- interacted in particular ways.  And because of chemical processes involving acids and bases, certain kinds of chemicals can begin to self-organize into larger structures, li

Rejected preview:



Human: Why did cells originally combine together to create life?

Assistant: Cells combine because they benefit from cooperation, since they can have less competition for resources by working together.


## 6. Convert Data to DPO Format

Separate each conversation pair into a shared prompt, preferred response, and rejected response.


In [6]:
def split_prompt(example):
    """
    Convert:
        chosen   = prompt + assistant_response
        rejected = prompt + assistant_response

    Into:
        prompt
        chosen
        rejected
    """

    chosen = example["chosen"]
    rejected = example["rejected"]

    marker = "\n\nAssistant:"

    split_idx = chosen.rfind(marker)

    if split_idx == -1:
        return {
            "prompt": "",
            "chosen": chosen,
            "rejected": rejected,
        }

    prompt = chosen[: split_idx + len(marker)]

    chosen_response = chosen[split_idx + len(marker):]
    rejected_response = rejected[split_idx + len(marker):]

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }


train_data = train_data.map(split_prompt)
test_data = test_data.map(split_prompt)

print("=" * 60)
print("DPO DATASET READY")
print("=" * 60)

print("Columns:", train_data.column_names)

print("\nPrompt:\n")
print(train_data[0]["prompt"][:300])

print("\nChosen response:\n")
print(train_data[0]["chosen"])

print("\nRejected response:\n")
print(train_data[0]["rejected"])


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/199 [00:00<?, ? examples/s]

DPO DATASET READY
Columns: ['chosen', 'rejected', 'prompt']

Prompt:



Human: Why did cells originally combine together to create life?

Assistant:

Chosen response:

 Because their simple components -- chemicals -- interacted in particular ways.  And because of chemical processes involving acids and bases, certain kinds of chemicals can begin to self-organize into larger structures, like membrane-bounded compartments.  And it’s from those compartments that life eventually emerged.

Rejected response:

 Cells combine because they benefit from cooperation, since they can have less competition for resources by working together.


## 7. Load the Tokenizer

Load the SmolLM2-360M tokenizer and configure the EOS token for padding when necessary.


In [7]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# SmolLM2 does not define a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("TOKENIZER READY")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Max length:", tokenizer.model_max_length)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

TOKENIZER READY
Model: HuggingFaceTB/SmolLM2-360M
Vocabulary size: 49152
EOS token: <|endoftext|>
PAD token: <|endoftext|>
Max length: 8192


## 8. Load the Base Model

Load `SmolLM2-360M` directly in FP32. The original development notebook first attempted FP16 training, which produced a gradient-unscaling error. The final reproducible configuration therefore uses full precision.


In [8]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

model.to(device)

print("=" * 60)
print("BASE MODEL READY")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Class:", model.__class__.__name__)
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nParameters: {total_params:,}")
print(f"Parameters (millions): {total_params/1e6:.1f}M")

if torch.cuda.is_available():
    print(f"\nGPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

BASE MODEL READY
Model: HuggingFaceTB/SmolLM2-360M
Class: LlamaForCausalLM
Device: cuda:0
Data type: torch.float32

Parameters: 361,821,120
Parameters (millions): 361.8M

GPU memory: 1.35 GB


## 9. DPO Training Configuration

Configure the final DPO experiment: one epoch, learning rate `1e-6`, batch size 1, gradient accumulation of 8, DPO beta `0.1`, maximum sequence length 512, and maximum prompt length 256. Mixed precision is disabled to match the final working FP32 setup.


In [9]:
dpo_config = DPOConfig(
    output_dir="./smollm2_dpo",

    num_train_epochs=1,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    max_length=512,
    max_prompt_length=256,

    beta=0.1,

    fp16=False,
    bf16=False,

    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="none",

    seed=SEED,
)

print("=" * 60)
print("DPO CONFIGURATION")
print("=" * 60)

print("Epochs:", dpo_config.num_train_epochs)
print("Learning rate:", dpo_config.learning_rate)
print("Batch size:", dpo_config.per_device_train_batch_size)
print("Gradient accumulation:", dpo_config.gradient_accumulation_steps)
print("Beta:", dpo_config.beta)
print("Max length:", dpo_config.max_length)
print("Max prompt length:", dpo_config.max_prompt_length)
print("FP16:", dpo_config.fp16)
print("BF16:", dpo_config.bf16)


DPO CONFIGURATION
Epochs: 1
Learning rate: 1e-06
Batch size: 1
Gradient accumulation: 8
Beta: 0.1
Max length: 512
Max prompt length: 256
FP16: False
BF16: False


## 10. Create the DPO Trainer

Create the TRL `DPOTrainer` using the policy model, tokenizer, sampled preference dataset, and final training configuration. With `ref_model=None`, TRL manages the reference-policy behavior required by DPO.


In [10]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    processing_class=tokenizer,
    train_dataset=train_data,
)

print("=" * 60)
print("DPO TRAINER READY")
print("=" * 60)

print("Training examples :", len(trainer.train_dataset))
print("Model class       :", trainer.model.__class__.__name__)
print("Model dtype       :", next(trainer.model.parameters()).dtype)
print("Device            :", next(trainer.model.parameters()).device)
print("\nDPOTrainer created successfully")


Extracting prompt in train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

DPO TRAINER READY
Training examples : 1000
Model class       : LlamaForCausalLM
Model dtype       : torch.float32
Device            : cuda:0

DPOTrainer created successfully


## 11. Train the DPO Model

Optimize the model on the sampled preference pairs using the final DPO configuration.


In [11]:
train_result = trainer.train()

print("\n" + "=" * 60)
print("DPO TRAINING COMPLETE")
print("=" * 60)

print(train_result)


Step,Training Loss
10,0.693200
20,0.693100
30,0.693400
40,0.692100
50,0.692300
60,0.692900
70,0.692900
80,0.692700
90,0.691500
100,0.691300



DPO TRAINING COMPLETE
TrainOutput(global_step=125, training_loss=0.6924661197662354, metrics={'train_runtime': 563.7544, 'train_samples_per_second': 1.774, 'train_steps_per_second': 0.222, 'total_flos': 0.0, 'train_loss': 0.6924661197662354, 'epoch': 1.0})


## 12. Save the DPO Model

Save the trained DPO model and tokenizer for reproducibility and later evaluation.


In [12]:
trainer.save_model("./smollm2_dpo")
tokenizer.save_pretrained("./smollm2_dpo")

print("=" * 60)
print("DPO MODEL SAVED")
print("=" * 60)

print("Directory: ./smollm2_dpo")
print("✓ Model saved")
print("✓ Tokenizer saved")


DPO MODEL SAVED
Directory: ./smollm2_dpo
✓ Model saved
✓ Tokenizer saved


## 13. Preference-Scoring Function

Define a mean response log-probability function so chosen and rejected responses can be ranked consistently.


In [13]:
import torch

def score_response(model, tokenizer, prompt, response):
    """
    Returns the mean log-probability of the assistant response
    conditioned on the prompt.
    """

    full_text = prompt + response

    full = tokenizer(full_text, return_tensors="pt")
    prompt_tokens = tokenizer(prompt, return_tensors="pt")

    input_ids = full.input_ids.to(model.device)
    attention_mask = full.attention_mask.to(model.device)

    prompt_len = prompt_tokens.input_ids.shape[1]

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    logits = outputs.logits[:, :-1]
    labels = input_ids[:, 1:]

    log_probs = torch.log_softmax(logits, dim=-1)

    token_log_probs = log_probs.gather(
        -1,
        labels.unsqueeze(-1)
    ).squeeze(-1)

    # Score only assistant response tokens
    response_log_probs = token_log_probs[:, prompt_len - 1:]

    return response_log_probs.mean().item()


print("✓ score_response() ready")


✓ score_response() ready


## 14. Final Held-Out Evaluation

Evaluate the trained DPO model on the 199 held-out preference pairs and report preference accuracy and score-margin statistics.


In [14]:
from tqdm.auto import tqdm
import numpy as np

correct = 0
margins = []

for example in tqdm(test_data, desc="Evaluating DPO Model"):

    chosen_score = score_response(
        model=model,
        tokenizer=tokenizer,
        prompt=example["prompt"],
        response=example["chosen"],
    )

    rejected_score = score_response(
        model=model,
        tokenizer=tokenizer,
        prompt=example["prompt"],
        response=example["rejected"],
    )

    margin = chosen_score - rejected_score
    margins.append(margin)

    if chosen_score > rejected_score:
        correct += 1

accuracy = correct / len(test_data)

print("=" * 60)
print("DPO MODEL — HELD-OUT PREFERENCE EVALUATION")
print("=" * 60)

print(f"\nTest pairs: {len(test_data)}")

print("\nPreference accuracy:")
print(f"{correct}/{len(test_data)} ({accuracy*100:.2f}%)")

print("\nPreference margin (chosen - rejected):")
print(f"Mean:   {np.mean(margins):.4f}")
print(f"Median: {np.median(margins):.4f}")

print("\nMargin range:")
print(f"Minimum: {np.min(margins):.4f}")
print(f"Maximum: {np.max(margins):.4f}")


Evaluating DPO Model:   0%|          | 0/199 [00:00<?, ?it/s]

DPO MODEL — HELD-OUT PREFERENCE EVALUATION

Test pairs: 199

Preference accuracy:
105/199 (52.76%)

Preference margin (chosen - rejected):
Mean:   0.0804
Median: 0.0411

Margin range:
Minimum: -5.8673
Maximum: 3.0826
